# 14 · Validation rigor & out-of-sample diagnostics

Beyond a single forward walk: **purged combinatorial CV** probes many train/test paths, **MDA importance** ranks features *out of sample*, and the **diagnostics** layer asks how predictive a signal is and for how long. Everything stays causal — no label leaks across a fold boundary.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from quantkit import backtest as B, models as MD, diagnostics as DG
from quantkit.labels import forward_return

# Synthetic cross-section with a real momentum edge (no network). On real data use
# quantkit.data.price_panel + quantkit.features; every call below is identical.
rng = np.random.default_rng(7)
dates = pd.bdate_range("2016-01-01", periods=700)
assets = [f"A{i:02d}" for i in range(25)]
mkt = rng.standard_normal((len(dates), 1)) * 0.01
prices = pd.DataFrame(
    100 * np.exp(np.cumsum(mkt + 0.008 * rng.standard_normal((len(dates), 25)), axis=0)),
    index=dates, columns=assets,
)
mom = prices / prices.shift(63) - 1.0
z = (mom - mom.rolling(126).mean()) / mom.rolling(126).std()
X, y = MD.make_design({"mom": mom, "z": z}, forward_return(prices, 5))
print("design:", X.shape, "| folds source dates:", len(dates))

design: (12675, 2) | folds source dates: 700


## Purged combinatorial CV (López de Prado)

Every size-`k` combination of time blocks becomes a test set, with a two-sided purge + embargo. Test blocks may sit in the interior of the timeline.

In [2]:
cpcv = B.combinatorial_purged(dates, n_groups=6, k_test=2, horizon=5, embargo=3)
print(f'{len(cpcv)} folds = C(6,2); all purged: {all(B.is_purged(f, dates, 5, 3) for f in cpcv)}')
interior = [f for f in cpcv if f.train[0] < f.test[0] and f.train[-1] > f.test[-1]]
print('example interior fold:', interior[0])

15 folds = C(6,2); all purged: True
example interior fold: Fold(train=2016-01-01..2018-09-06 (n=458), test=2016-06-14..2017-05-05 (n=234))


## MDA feature importance (out of sample)

Fit per fold, permute one feature in the *test* block, measure the rank-IC drop. A noise feature scores ~0; the real driver scores high. Deterministic given a seed.

In [3]:
folds = B.walk_forward(dates, train=200, test=42, horizon=5, embargo=3)
imp = MD.mda_importance(MD.ridge, X, y, folds, n_repeats=5, random_state=0)
imp.round(4)

,importance,std,n
feature,,,
mom,0.0234,0.0713,55.0
z,-0.0087,0.0734,55.0


## IC decay

How fast does the signal's edge fade with the forecast horizon?

In [4]:
decay = DG.ic_decay(z, prices, horizons=[1, 5, 10, 21, 63])
fig = go.Figure(go.Scatter(x=decay.index, y=decay.to_numpy(), mode='lines+markers'))
fig.update_layout(title='IC decay', xaxis_title='horizon (bars)', yaxis_title='rank IC',
                  template='plotly_white', height=360)
fig.add_hline(y=0, line={'dash': 'dot', 'color': 'rgba(0,0,0,.3)'})
fig

## Factor exposure & capacity

Attribution (is the edge just market beta?) and how much capital the trade list absorbs before turnover in thin names binds.

In [5]:
strat_ret = y.groupby(y.index.get_level_values('date')).mean()  # toy strategy P&L
factor = prices.pct_change(fill_method=None).mean(axis=1).rename('MKT')
exp = DG.factor_exposure(strat_ret, factor.to_frame()).round(4)
print(exp.to_string(), '\nR^2 =', round(exp.attrs['r_squared'], 4))
w = pd.DataFrame(1.0 / 25, index=dates, columns=assets)
adv = pd.Series(5e6, index=assets)
cap = DG.capacity(w.iloc[::21], adv, participation=0.1)  # at monthly rebalances
print('capacity ($, monthly):', cap.dropna().round(0).head().tolist())

         beta  t_stat
alpha -0.0042 -4.1722
MKT    0.0607  0.5768 
R^2 = 0.0007
capacity ($, monthly): [12500000.0]


**Takeaway.** CPCV + MDA keep the model honest before any capital is risked; IC decay sets the rebalance horizon; exposure/capacity say whether the edge is real and how large it can run.